## 1 - Dataset Acquisition and Initial Setup

In this first step, we are downloading the CelebDF dataset, the **3rd** version.

In [ ]:
import cv2
import glob
import os
import pandas as pd
from pathlib import Path
from tqdm import tqdm

In [2]:
base_path = "/Volumes/LUCIA'S HD/Celeb-DF-v3/"
folders = os.listdir(base_path)
print("Contents inside", base_path, ": ")
print('\n'.join([f"- {f}" for f in folders]))

Contents inside /Volumes/LUCIA'S HD/Celeb-DF-v3/ : 
- Celeb-real
- ._Celeb-real
- Celeb-synthesis
- ._Celeb-synthesis
- List_of_testing_videos.txt
- ._List_of_testing_videos.txt
- YouTube-real
- ._YouTube-real


In [3]:
for folder in folders:
    if folder.startswith('._'):
        continue
        
    folder_path = os.path.join(base_path, folder)

    if os.path.isdir(folder_path):
        try:
            files = [f for f in os.listdir(folder_path) if not f.startswith('._')]
            if files:
                print(f"Directory '{folder}': {len(files)} files (e.g., {files[0]})")
            else:
                print(f"Directory '{folder}': Empty")
        except Exception as e:
            print(f"Error accessing {folder}: {e}")
    else:
        print(f"File found: {folder}")

Directory 'Celeb-real': 590 files (e.g., id0_0000.mp4)
Directory 'Celeb-synthesis': 3 files (e.g., FaceReenact)
File found: List_of_testing_videos.txt
Directory 'YouTube-real': 300 files (e.g., 00000.mp4)


In [4]:
synthesis_path = r"/Volumes/LUCIA'S HD/Celeb-DF-v3/Celeb-synthesis"
all_fake_videos = []

print("Looking for fake videos...")

for root, dirs, files in os.walk(synthesis_path):
    vids = [os.path.join(root, f) for f in files 
            if f.endswith(('.mp4', '.avi')) and not f.startswith('._')]
    all_fake_videos.extend(vids)

print(f"\n--- FAKE VIDEO ---")
print(f"Total: {len(all_fake_videos)}")

if all_fake_videos:
    print(f"Example path: {all_fake_videos[0]}")
    counts = {}
    for path in all_fake_videos:
        parts = path.split(os.sep)
        idx = parts.index('Celeb-synthesis')
        category = parts[idx + 1]
        counts[category] = counts.get(category, 0) + 1
    
    for cat, count in counts.items():
        print(f"- {cat}: {count} video")

Looking for fake videos...

--- FAKE VIDEO ---
Total: 53196
Example path: /Volumes/LUCIA'S HD/Celeb-DF-v3/Celeb-synthesis/FaceReenact/DaGAN/id0_id16_0001.mp4
- FaceReenact: 13632 video
- FaceSwap: 19285 video
- TalkingFace: 20279 video


In [ ]:
real_folders = ['Celeb-real', 'YouTube-real']

print(f"--- REAL VIDEO ---")

for folder in real_folders:
    folder_path = os.path.join(base_path, folder)
    print(f"\nDirectory: {folder}")
    
    if os.path.exists(folder_path):
        all_items = [f for f in os.listdir(folder_path) if not f.startswith('._')]
        
        vids = [f for f in all_items if f.endswith(('.mp4', '.avi'))]
        subdirs = [f for f in all_items if os.path.isdir(os.path.join(folder_path, f))]
        
        print(f"  - Total elements: {len(all_items)}")
        print(f"  - Total video: {len(vids)}")
        print(f"  - Subdirectory: {len(subdirs)}")
        
        if vids:
            print(f"  - Example video: {vids[:3]}")
        if subdirs:
            print(f"  - Example subdirectory: {subdirs[:3]}")
            first_sub = os.path.join(folder_path, subdirs[0])
            inner_vids = [f for f in os.listdir(first_sub) if f.endswith(('.mp4', '.avi'))]
            print(f"  - Video inside the first subdirectory ({subdirs[0]}): {len(inner_vids)}")
    else:
        print(f"  Direcotry not founded: {folder_path}")

--- REAL VIDEO ---

📂 Directory: Celeb-real
  - Total elements: 590
  - Total video: 590
  - Subdirectory: 0
  - Example video: ['id0_0000.mp4', 'id0_0001.mp4', 'id0_0002.mp4']

📂 Directory: YouTube-real
  - Total elements: 300
  - Total video: 300
  - Subdirectory: 0
  - Example video: ['00000.mp4', '00001.mp4', '00002.mp4']


## 2 - Pandas DataFrame

This section creates a single pandas DataFrame containing all videos from the Celeb-DF-v3 dataset.

**Columns:**
- `video`: the video filename (e.g., with `.mp4` extension).
- `full_path`: the complete file path to the video on the disk.
- `label`: 0 = real/original, 1 = fake/synthesis.
- `dataset`: the name of the dataset (e.g., "Celeb-DF-v3").
- `category`: the specific subfolder or subset category (e.g., 'Celeb-real', 'YouTube-real', or synthesis sub-categories).
- `method`: the specific deepfake generation method used, or "original" for real videos.
- `target`: ID of the target subject (the person whose face is being replaced or modified).
- `source`: ID of the source subject (the person providing the new face). For real videos, this is identical to the target.
- `sequence`: the video sequence identifier extracted from the filename, or "original" for real videos.

In [ ]:
data = []

real_folders = ['Celeb-real', 'YouTube-real']
for folder in real_folders:
    folder_path = os.path.join(base_path, folder)
    if not os.path.exists(folder_path): continue
    
    for video in os.listdir(folder_path):
        if video.endswith('.mp4') and not video.startswith('._'):
            name = video.replace('.mp4', '')
            target = name.split('_')[0] if 'id' in name else name
            
            data.append({
                "video": video,
                "label": 0,
                "dataset": "Celeb-DF-v3",
                "category": folder,
                "method": "original",
                "target": target,
                "source": target, # Reale: Source = Target
                "sequence": "original",
                "full_path": os.path.join(folder_path, video)
            })

synthesis_path = os.path.join(base_path, 'Celeb-synthesis')
if os.path.exists(synthesis_path):
    for root, dirs, files in os.walk(synthesis_path):
        for video in files:
            if video.endswith('.mp4') and not video.startswith('._'):
                name = video.replace('.mp4', '')
                parts = name.split('_')
   
                path_parts = root.split(os.sep)
                idx = path_parts.index('Celeb-synthesis')
                category = path_parts[idx + 1] if len(path_parts) > idx + 1 else "Unknown"
                method = path_parts[idx + 2] if len(path_parts) > idx + 2 else "Unknown"

                source = parts[0] if len(parts) > 0 else "Unknown"
                sequence = parts[1] if len(parts) > 1 else "Unknown"

                target = "Unknown"
                for p in parts:
                    if p.startswith('id') and p != source:
                        target = p
                        break
                
                if target == "Unknown":
                    target = source

                data.append({
                    "video": video,
                    "label": 1,
                    "dataset": "Celeb-DF-v3",
                    "category": category,
                    "method": method,
                    "target": target,
                    "source": source,
                    "sequence": sequence,
                    "full_path": os.path.join(root, video)
                })

df_celeb = pd.DataFrame(data)

print("--- CELEB-DF-V3 DATAFRAME ---")
display(df_celeb.sample(15))

--- CELEB-DF-V3 DATAFRAME ---


,video,label,dataset,category,method,target,source,sequence,full_path
26382,id21_id32_0009.mp4,1,Celeb-DF-v3,FaceSwap,InSwapper,id32,id21,id32,/Volumes/LUCIA'S HD/Celeb-DF-v3/Celeb-synthesi...
31882,id0_id28_0002.mp4,1,Celeb-DF-v3,FaceSwap,UniFace,id28,id0,id28,/Volumes/LUCIA'S HD/Celeb-DF-v3/Celeb-synthesi...
22582,id23_id30_0000.mp4,1,Celeb-DF-v3,FaceSwap,GHOST,id30,id23,id30,/Volumes/LUCIA'S HD/Celeb-DF-v3/Celeb-synthesi...
1640,id2_id1_0002.mp4,1,Celeb-DF-v3,FaceReenact,DaGAN,id1,id2,id1,/Volumes/LUCIA'S HD/Celeb-DF-v3/Celeb-synthesi...
53830,id60_0008_test_id07414_8Ii7TAg4xlE.mp4,1,Celeb-DF-v3,TalkingFace,SadTalker,id07414,id60,0008,/Volumes/LUCIA'S HD/Celeb-DF-v3/Celeb-synthesi...
14108,id49_id58_0008.mp4,1,Celeb-DF-v3,FaceReenact,TPSMM,id58,id49,id58,/Volumes/LUCIA'S HD/Celeb-DF-v3/Celeb-synthesi...
28202,id20_id17_0002.mp4,1,Celeb-DF-v3,FaceSwap,MobileFaceSwap,id17,id20,id17,/Volumes/LUCIA'S HD/Celeb-DF-v3/Celeb-synthesi...
712,00122.mp4,0,Celeb-DF-v3,YouTube-real,original,00122,00122,original,/Volumes/LUCIA'S HD/Celeb-DF-v3/YouTube-real/0...
6327,id4_id23_0006.mp4,1,Celeb-DF-v3,FaceReenact,HyperReenact,id23,id4,id23,/Volumes/LUCIA'S HD/Celeb-DF-v3/Celeb-synthesi...
40524,id27_0009_test_id05055_9xBtaJ-hpQY.mp4,1,Celeb-DF-v3,TalkingFace,EDTalk,id05055,id27,0009,/Volumes/LUCIA'S HD/Celeb-DF-v3/Celeb-synthesi...


### 2.1 - Sanity Check

Quick checks to ensure the dataset is loaded correctly and to understand its composition:

- **Total number of videos:** Overall count of files found on the drive.
- **Label distribution:** Balance between real (0) and fake (1) videos.
- **Method distribution:** The top AI manipulation methods used to generate the fakes.
- **Category distribution:** Breakdown by subset (e.g., Celeb-real, YouTube-real, Celeb-synthesis).
- **Identity analysis:** Counts of unique target and source IDs, including the overlap of identities present in both real and fake sets.

In [7]:
print(f"--- CELEB-DF-V3 AUDIT ---")
print(f"Total videos found on HDD: {len(df_celeb)}")

print("\nLabel Distribution:")
print(df_celeb["label"].value_counts().rename({0: '0 (Real)', 1: '1 (Fake)'}))

print("\nAI Method Distribution (Top 10):")
print(df_celeb["method"].value_counts().head(10))

print("\nCategory Distribution:")
print(df_celeb["category"].value_counts())

print("\nIdentity Analysis (Target-based):")
unique_targets = df_celeb['target'].nunique()
print(f"Total unique Target identities: {unique_targets}")

real_targets = set(df_celeb[df_celeb['label'] == 0]['target'])
fake_targets = set(df_celeb[df_celeb['label'] == 1]['target'])
overlap = real_targets.intersection(fake_targets)

print(f"Targets present in both Real and Fake: {len(overlap)}")
print(f"Targets only in Real: {len(real_targets - fake_targets)}")
print(f"Targets only in Fake: {len(fake_targets - real_targets)}")

if df_celeb['label'].sum() > 0:
    unique_sources = df_celeb[df_celeb['label'] == 1]['source'].nunique()
    print(f"Unique Sources used for fakes: {unique_sources}")

--- CELEB-DF-V3 AUDIT ---
Total videos found on HDD: 54086

Label Distribution:
label
1 (Fake)    53196
0 (Real)      890
Name: count, dtype: int64

AI Method Distribution (Top 10):
method
Celeb-DF-v2       5639
SadTalker         2950
FLOAT             2950
AniTalker         2950
IP_LAP            2935
EchoMimic         2885
EDTalk            2865
Real3DPortrait    2744
SimSwap           1953
MobileFaceSwap    1953
Name: count, dtype: int64

Category Distribution:
category
TalkingFace     20279
FaceSwap        19285
FaceReenact     13632
Celeb-real        590
YouTube-real      300
Name: count, dtype: int64

Identity Analysis (Target-based):
Total unique Target identities: 476
Targets present in both Real and Fake: 57
Targets only in Real: 302
Targets only in Fake: 117
Unique Sources used for fakes: 59


### 2.2  - Distribution of Fake Videos per Target

Analyzes how many fake videos exist per target identity to identify if some identities dominate the fake samples

In [8]:
fake_df = df_celeb[df_celeb["label"] == 1]
per_target = fake_df.groupby("target").size()
print("\n--- FAKE PER TARGET STATISTICS ---")
print(per_target.describe())


--- FAKE PER TARGET STATISTICS ---
count     174.000000
mean      305.724138
std       286.455746
min         7.000000
25%       123.750000
50%       218.000000
75%       370.500000
max      1318.000000
dtype: float64


### 2.3 - Distribution Of Methods per Target (Check Variability):

Creates a pivot table showing how many videos of each manipulation method exist per target to verify that each identity has a representative set of manipulation methods and to detect identities with too few or missing manipulation types

In [9]:
pivot_celeb = pd.pivot_table(
    df_celeb,
    index="target",
    columns="method",
    values="video",
    aggfunc="count",
    fill_value=0
)

print("\n--- TOP 20 TARGETS BY METHOD COVERAGE ---")
if 'original' in pivot_celeb.columns:
    display(pivot_celeb.sort_values(by='original', ascending=False).head(20))
else:
    display(pivot_celeb.sample(20))


--- TOP 20 TARGETS BY METHOD COVERAGE ---


method,AniTalker,BlendFace,Celeb-DF-v2,DaGAN,EDTalk,EchoMimic,FLOAT,FSRT,GHOST,HifiFace,...,LIA,LivePortrait,MCNET,MobileFaceSwap,Real3DPortrait,SadTalker,SimSwap,TPSMM,UniFace,original
target,,,,,,,,,,,,,,,,,,,,,
id13,0,6,16,6,0,0,0,6,6,6,...,6,6,6,6,0,0,6,6,6,16
id16,0,48,145,48,0,0,0,48,48,48,...,48,48,48,48,0,0,48,48,48,14
id11,0,6,16,5,0,0,0,5,6,6,...,6,6,5,6,0,0,6,6,6,11
id25,0,17,71,17,0,0,0,17,17,17,...,17,17,17,17,0,0,17,17,17,11
id9,0,55,168,55,0,0,0,55,55,55,...,55,55,55,55,0,0,55,55,55,10
id27,0,25,64,20,0,0,0,20,25,25,...,20,20,20,25,0,0,25,20,25,10
id36,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,10
id35,0,70,189,70,0,0,0,70,70,70,...,70,70,70,70,0,0,70,70,70,10
id34,0,39,108,39,0,0,0,39,39,39,...,39,39,39,39,0,0,39,39,39,10


### 2.4 - Check Label Balance

Checks the ratio of real to fake videos across the entire dataset to understand dataset imbalance

In [10]:
print("Normalize label ditribution:")
print(df_celeb["label"].value_counts(normalize=True))

Normalize label ditribution:
label
1    0.983545
0    0.016455
Name: proportion, dtype: float64


## 3 - Metadata Extraction (OpenCV)

In this section, we use **OpenCV (`cv2`)** to scan through the entire Celeb-DF-v3 dataset and extract key metadata:
- `total_frames`
- `fps`
- `duration_sec`
- `resolution`

In [ ]:
tqdm.pandas(desc="Extracting Celeb-DF-v3 metadata")

def get_video_metadata(video_path):
    try:
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            return pd.Series([None, None, None, None])
    
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        fps = cap.get(cv2.CAP_PROP_FPS)
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        
        duration = total_frames / fps if fps > 0 else 0
        
        cap.release()
        return pd.Series([total_frames, fps, duration, f"{width}x{height}"])
    
    except Exception as e:
        return pd.Series([None, None, None, None])

print(f"Scanning {len(df_celeb)} videos. This will take a few minutes...")

df_celeb[['total_frames', 'fps', 'duration_sec', 'resolution']] = \
    df_celeb['full_path'].progress_apply(get_video_metadata)

print("\nMetadata extraction complete!")

display(df_celeb[['video', 'total_frames', 'duration_sec', 'resolution', 'method']].sample(20))

Scanning 54086 videos. This will take a few minutes...


Extracting Celeb-DF-v3 metadata: 100%|██████████| 54086/54086 [09:49<00:00, 91.75it/s] 


✅ Metadata extraction complete!


,video,total_frames,duration_sec,resolution,method
53726,id59_0008_test_id01228_wQ0cqLka-oQ.mp4,145,5.800000,256x240,SadTalker
14405,id61_id60_0002.mp4,339,11.300000,256x256,TPSMM
7747,id34_id26_0003.mp4,248,8.266667,256x256,LIA
35845,id49_0007_test_id03839_pX7kYiDgFMA.mp4,142,5.680000,256x256,AniTalker
22404,id20_id30_0009.mp4,334,11.133333,666x500,GHOST
39714,id10_0004_test_id02057_3Ugj7nq-mlg.mp4,131,5.240000,256x256,EDTalk
12315,id54_id51_0006.mp4,436,14.572193,256x256,MCNET
52601,id38_0004_test_id00419_WOVrVh7U0j8.mp4,140,5.600000,256x256,SadTalker
30860,id31_id9_0005.mp4,349,11.633333,850x472,SimSwap
29571,id51_id54_0004.mp4,447,15.413793,856x478,MobileFaceSwap


In [12]:
print("\n--- VIDEO DURATION STATISTICS (in seconds) ---")
print(df_celeb['duration_sec'].describe())

print("\n--- VIDEO FRAMES STATISTICS ---")
print(df_celeb['total_frames'].describe())

display(df_celeb.sample(20))


--- VIDEO DURATION STATISTICS (in seconds) ---
count    54086.000000
mean        10.308581
std          4.042332
min          0.033333
25%          6.720000
50%         10.400000
75%         13.000000
max         36.950000
Name: duration_sec, dtype: float64

--- VIDEO FRAMES STATISTICS ---
count    54086.000000
mean       287.973727
std        122.839820
min          1.000000
25%        170.000000
50%        306.000000
75%        365.000000
max        741.000000
Name: total_frames, dtype: float64


,video,label,dataset,category,method,target,source,sequence,full_path,total_frames,fps,duration_sec,resolution
40670,id30_0000_test_id04232__k0hgacT2zc.mp4,1,Celeb-DF-v3,TalkingFace,EDTalk,id04232,id30,0000,/Volumes/LUCIA'S HD/Celeb-DF-v3/Celeb-synthesi...,241,25.0,9.640000,256x256
22994,id31_id16_0004.mp4,1,Celeb-DF-v3,FaceSwap,GHOST,id16,id31,id16,/Volumes/LUCIA'S HD/Celeb-DF-v3/Celeb-synthesi...,465,30.0,15.500000,850x472
1145,id20_id1_0002.mp4,1,Celeb-DF-v3,FaceReenact,DaGAN,id1,id20,id1,/Volumes/LUCIA'S HD/Celeb-DF-v3/Celeb-synthesi...,399,30.0,13.300000,256x256
24584,id24_id26_0005.mp4,1,Celeb-DF-v3,FaceSwap,HifiFace,id26,id24,id26,/Volumes/LUCIA'S HD/Celeb-DF-v3/Celeb-synthesi...,337,30.0,11.233333,848x478
32381,id24_id23_0009.mp4,1,Celeb-DF-v3,FaceSwap,UniFace,id23,id24,id23,/Volumes/LUCIA'S HD/Celeb-DF-v3/Celeb-synthesi...,310,30.0,10.333333,850x472
51183,id0_0009_test_id03347_DhBctdcCSzI.mp4,1,Celeb-DF-v3,TalkingFace,SadTalker,id03347,id0,0009,/Volumes/LUCIA'S HD/Celeb-DF-v3/Celeb-synthesi...,241,25.0,9.640000,256x256
25897,id6_id1_0001.mp4,1,Celeb-DF-v3,FaceSwap,HifiFace,id1,id6,id1,/Volumes/LUCIA'S HD/Celeb-DF-v3/Celeb-synthesi...,323,30.0,10.766667,942x500
53899,id6_0002_test_id05124_GlQBVzhcSzs.mp4,1,Celeb-DF-v3,TalkingFace,SadTalker,id05124,id6,0002,/Volumes/LUCIA'S HD/Celeb-DF-v3/Celeb-synthesi...,142,25.0,5.680000,256x256
20646,id46_id47_0002.mp4,1,Celeb-DF-v3,FaceSwap,Celeb-DF-v2,id47,id46,id47,/Volumes/LUCIA'S HD/Celeb-DF-v3/Celeb-synthesi...,313,30.0,10.433333,670x500
42180,id5_0001_test_id06310_rax-2smxzpk.mp4,1,Celeb-DF-v3,TalkingFace,EDTalk,id06310,id5,0001,/Volumes/LUCIA'S HD/Celeb-DF-v3/Celeb-synthesi...,142,25.0,5.680000,256x256


## 4 - Saving Processed Data

To conclude this notebook, we save the fully cleaned and processed DataFrame to a local CSV file (`./processed_videos/celeb_videos`). This ensures our prepared dataset is safely stored and ready to be directly loaded into the next notebook of our pipeline without needing to re-run the preprocessing steps.

In [13]:
print("--- SAVING DATAFRAME ---")

output_dir = "./processed_images"
os.makedirs(output_dir, exist_ok=True)

save_path = os.path.join(output_dir, "celeb_videos.csv")
df_celeb.to_csv(save_path, index=False)

print(f"Data succesfully saved to: {save_path}")

--- SAVING DATAFRAME ---
Data succesfully saved to: ./processed_images/celeb_videos.csv
